# Programación orientada a objetos con Python
## Video 5. Métodos especiales (dunder): `__repr__`, `__str__`, `__eq__`, `__lt__`, `__add__`, `__len__`

**Hilo conductor de la serie:** construiremos objetos matemáticos (fracciones, vectores y polinomios) y los iremos ampliando de video en video.

**Al terminar este video podrás:**
1. Distinguir entre `__repr__` (representación formal para desarrollo) y `__str__` (formato legible para `print`).
2. Definir la igualdad matemática por valor (`__eq__`) y el orden estricto (`__lt__`) sin ambigüedades.
3. Sobrecargar el operador de suma (`__add__`) empleando `NotImplemented` cuando los tipos son incompatibles.
4. Implementar `__len__` en objetos compuestos y relacionarlo con la dimensión o la cantidad de coeficientes.

## 1. Planteamiento del problema

**Definición.** En el cuerpo ordenado de los números racionales $(\mathbb{Q}, +, \cdot, <)$, dos fracciones normalizadas $(p, q)$ y $(r, s)$ con $q, s > 0$ satisfacen:

$$(p, q) = (r, s) \iff (p = r \text{ y } q = s), \qquad (p, q) < (r, s) \iff p\,s < q\,r.$$

En el espacio euclidiano $\mathbb{R}^n$, dos vectores $u = (u_1, \dots, u_n)$ y $v = (v_1, \dots, v_n)$ de igual dimensión $n$ tienen longitud $n$ y se suman componente a componente:

$$u + v = (u_1 + v_1, \dots, u_n + v_n).$$

**Problema: integración con los operadores y funciones integradas de Python.**

- **Entrada:** instancias válidas de `Fraccion`, `Vector` y `Polinomio`.
- **Salida:** clases equipadas con métodos especiales para que funcionen de manera natural expresiones como `repr(x)`, `print(x)`, `x == y`, `x < y`, `sorted(lista)`, `x + y` y `len(x)`.

## 2. Representación en texto: `__repr__` y `__str__`

En Python, la palabra *dunder* proviene de doble guion bajo (*double underscore*). Estos métodos especiales permiten que nuestras clases respondan a las funciones y operadores del lenguaje:

- `__repr__(self)`: devuelve una cadena técnica e inequívoca destinada a quien programa; siempre que sea posible debe parecer una llamada válida al constructor, como `"Fraccion(3, 4)"`. Es la representación que muestran las listas, diccionarios y consolas interactivas.
- `__str__(self)`: devuelve una cadena amigable destinada a la lectura humana cuando se llama a `str(objeto)` o `print(objeto)`. Si una clase define `__repr__` pero no define `__str__`, Python utiliza `__repr__` como respaldo.

In [ ]:
from math import gcd, sqrt


class Fraccion:
    def __init__(self, numerador, denominador=1):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def __repr__(self):
        return f"Fraccion({self._numerador}, {self._denominador})"

    def __str__(self):
        if self._denominador == 1:
            return str(self._numerador)
        return f"{self._numerador}/{self._denominador}"


a = Fraccion(6, 8)
b = Fraccion(10, 2)
print("repr(a):", repr(a))
print("str(a): ", str(a))
print("print(b):", b)
print("Lista de fracciones (usa __repr__):", [a, b])

## 3. Igualdad (`__eq__`) y orden (`__lt__`)

Por defecto, si una clase no define `__eq__`, el operador `==` compara la **identidad** en memoria (`a is b`), de modo que dos fracciones creadas por separado con el mismo valor resultan distintas para `==`.

Al definir `__eq__(self, otra)` y `__lt__(self, otra)`:
- Si `otra` no pertenece a un tipo comparable (por ejemplo, una cadena de texto), se debe devolver la constante especial `NotImplemented`. De este modo Python intenta la comparación reflejada en el otro operando o lanza `TypeError` en el caso de `<`.
- Al definir `__lt__`, funciones como `sorted`, `min` y `max` ordenan colecciones de fracciones automáticamente.

In [ ]:
class FraccionSinIgualdad:
    def __init__(self, numerador, denominador):
        self.numerador = numerador
        self.denominador = denominador


print(
    "Sin __eq__, FraccionSinIgualdad(1, 2) == FraccionSinIgualdad(1, 2):",
    FraccionSinIgualdad(1, 2) == FraccionSinIgualdad(1, 2),
)


class Fraccion:
    def __init__(self, numerador, denominador=1):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def __repr__(self):
        return f"Fraccion({self._numerador}, {self._denominador})"

    def __str__(self):
        if self._denominador == 1:
            return str(self._numerador)
        return f"{self._numerador}/{self._denominador}"

    def __eq__(self, otra):
        if isinstance(otra, int):
            otra = Fraccion(otra, 1)
        elif not isinstance(otra, Fraccion):
            return NotImplemented
        return self._numerador == otra._numerador and self._denominador == otra._denominador

    def __lt__(self, otra):
        if isinstance(otra, int):
            otra = Fraccion(otra, 1)
        elif not isinstance(otra, Fraccion):
            return NotImplemented
        return self._numerador * otra._denominador < self._denominador * otra._numerador


print("Fraccion(1, 2) == Fraccion(2, 4):", Fraccion(1, 2) == Fraccion(2, 4))
print("Fraccion(6, 3) == 2:", Fraccion(6, 3) == 2)
desordenadas = [Fraccion(3, 4), Fraccion(-1, 2), Fraccion(2, 5), Fraccion(1, 3)]
print("Ordenadas de menor a mayor:", sorted(desordenadas))
print("Mínima:", min(desordenadas), "| Máxima:", max(desordenadas))

## 4. Operador de suma (`__add__`) y longitud (`__len__`)

El método `__add__(self, otro)` implementa la expresión binaria `a + b`:
- En `Fraccion`, devuelve una nueva instancia normalizada; si `otro` es un entero, lo convierte en fracción; si es de un tipo incompatible, devuelve `NotImplemented`.
- En `Polinomio`, suma los coeficientes término a término completando con ceros el polinomio de menor grado y normalizando los ceros finales resultantes si los términos principales se cancelan.

Por su parte, el método `__len__(self)` implementa la función integrada `len(objeto)` y debe devolver un entero no negativo: en un vector de $\mathbb{R}^n$ devuelve su dimensión $n$, y en un polinomio devuelve la cantidad de coeficientes almacenados.

In [ ]:
class Polinomio:
    variable = "x"

    def __init__(self, coeficientes):
        lista = list(coeficientes)
        if not lista:
            raise ValueError("La secuencia de coeficientes no puede estar vacía.")
        while len(lista) > 1 and lista[-1] == 0:
            lista.pop()
        self._coeficientes = tuple(lista)

    @property
    def coeficientes(self):
        return self._coeficientes

    @property
    def grado(self):
        return len(self._coeficientes) - 1

    def __repr__(self):
        return f"Polinomio({list(self._coeficientes)})"

    def __str__(self):
        return " + ".join(
            f"{coeficiente}*{self.variable}^{exponente}"
            for exponente, coeficiente in enumerate(self._coeficientes)
        )

    def __len__(self):
        return len(self._coeficientes)

    def __eq__(self, otro):
        if not isinstance(otro, Polinomio):
            return NotImplemented
        return self._coeficientes == otro._coeficientes

    def __add__(self, otro):
        if not isinstance(otro, Polinomio):
            return NotImplemented
        longitud_maxima = max(len(self._coeficientes), len(otro._coeficientes))
        suma = []
        for indice in range(longitud_maxima):
            a = self._coeficientes[indice] if indice < len(self._coeficientes) else 0
            b = otro._coeficientes[indice] if indice < len(otro._coeficientes) else 0
            suma.append(a + b)
        return Polinomio(suma)


p = Polinomio([1, 2, 3])
q = Polinomio([4, -1])
print("p =", p, "| len(p) =", len(p))
print("q =", q, "| len(q) =", len(q))
print("p + q =", p + q, "| repr:", repr(p + q))

### Tabla comparativa de métodos especiales

| Método especial | Invocado por | Valor de retorno esperado | Significado en nuestros objetos |
|---|---|---|---|
| `__repr__(self)` | `repr(x)` o consola | Cadena inequívoca | `"Fraccion(3, 4)"`, `"Polinomio([1, 2])"` |
| `__str__(self)` | `str(x)` o `print(x)` | Cadena legible | `"3/4"`, `"1*x^0 + 2*x^1"` |
| `__eq__(self, otro)` | `x == y`, `x != y` | Booleano o `NotImplemented` | Igualdad matemática de estado |
| `__lt__(self, otro)` | `x < y`, `sorted(...)` | Booleano o `NotImplemented` | Orden estricto en $\mathbb{Q}$ |
| `__add__(self, otro)` | `x + y` | Nuevo objeto o `NotImplemented` | Suma exacta en $\mathbb{Q}$, $\mathbb{R}^n$ o polinomios |
| `__len__(self)` | `len(x)` | Entero no negativo | Dimensión de `Vector` o número de coeficientes de `Polinomio` |

## 5. Ejercicios

**Ejercicio 1 (`Fraccion` con los cinco métodos especiales de aritmética y orden).** Construye la clase `Fraccion` integrando `__repr__`, `__str__`, `__eq__`, `__lt__` y `__add__`:
- **Entrada:** pares de enteros $(p, q)$ con $q \neq 0$, además de operaciones entre dos instancias de `Fraccion` o entre una `Fraccion` y un entero `int`.
- **Salida:** verifica que `Fraccion(1, 2) + Fraccion(1, 3) == Fraccion(5, 6)`, que `Fraccion(2, 3) + 1 == Fraccion(5, 3)` y que `sorted([Fraccion(1, 2), Fraccion(1, 3), Fraccion(3, 4)])` quede ordenada ascendentemente.

**Ejercicio 2 (clase `Vector` en $\mathbb{R}^n$ con `__repr__`, `__str__`, `__eq__`, `__add__` y `__len__`).** Diseña la clase `Vector` que reciba una secuencia no vacía de componentes reales en `__init__(self, componentes)` y las almacene en una tupla inmutable de flotantes `_componentes`:
- **Entrada:** dos instancias `u` y `v` de `Vector`.
- **Salida:** `len(u)` devuelve la dimensión $n$; `u == v` compara sus tuplas de componentes; `u + v` devuelve la suma componente a componente o lanza `ValueError` si `len(u) != len(v)` (y devuelve `NotImplemented` si `v` no es un `Vector`).

**Ejercicio 3 (`Polinomio` con cancelación de términos principales).** Comprueba y demuestra sobre la clase `Polinomio` de la sección anterior qué ocurre cuando dos polinomios de grado $2$ tienen coeficientes cuadráticos y lineales opuestos:
- **Entrada:** $P(x) = 1 + 2x + 3x^2$ y $Q(x) = 4 - 2x - 3x^2$.
- **Salida:** verifica que `P + Q == Polinomio([5])`, que `(P + Q).grado == 0` y que `len(P + Q) == 1`.

## 6. Soluciones

In [ ]:
# Ejercicio 1
class Fraccion:
    def __init__(self, numerador, denominador=1):
        if not isinstance(numerador, int) or not isinstance(denominador, int):
            raise TypeError("Numerador y denominador deben ser números enteros.")
        if denominador == 0:
            raise ValueError("El denominador no puede ser cero.")
        if denominador < 0:
            numerador, denominador = -numerador, -denominador
        divisor = gcd(numerador, denominador)
        self._numerador = numerador // divisor
        self._denominador = denominador // divisor

    @property
    def numerador(self):
        return self._numerador

    @property
    def denominador(self):
        return self._denominador

    def __repr__(self):
        return f"Fraccion({self._numerador}, {self._denominador})"

    def __str__(self):
        if self._denominador == 1:
            return str(self._numerador)
        return f"{self._numerador}/{self._denominador}"

    def __eq__(self, otra):
        if isinstance(otra, int):
            otra = Fraccion(otra, 1)
        elif not isinstance(otra, Fraccion):
            return NotImplemented
        return self._numerador == otra._numerador and self._denominador == otra._denominador

    def __lt__(self, otra):
        if isinstance(otra, int):
            otra = Fraccion(otra, 1)
        elif not isinstance(otra, Fraccion):
            return NotImplemented
        return self._numerador * otra._denominador < self._denominador * otra._numerador

    def __add__(self, otra):
        if isinstance(otra, int):
            otra = Fraccion(otra, 1)
        elif not isinstance(otra, Fraccion):
            return NotImplemented
        return Fraccion(
            self._numerador * otra._denominador + self._denominador * otra._numerador,
            self._denominador * otra._denominador,
        )


assert Fraccion(1, 2) + Fraccion(1, 3) == Fraccion(5, 6)
assert Fraccion(2, 3) + 1 == Fraccion(5, 3)
ordenadas = sorted([Fraccion(1, 2), Fraccion(1, 3), Fraccion(3, 4)])
print("1/2 + 1/3 =", Fraccion(1, 2) + Fraccion(1, 3))
print("2/3 + 1 =", Fraccion(2, 3) + 1)
print("Lista ordenada:", [str(item) for item in ordenadas])

In [ ]:
# Ejercicio 2
class Vector:
    def __init__(self, componentes):
        lista = [float(valor) for valor in componentes]
        if not lista:
            raise ValueError("Un vector debe tener al menos una componente.")
        self._componentes = tuple(lista)

    @property
    def componentes(self):
        return self._componentes

    def __len__(self):
        return len(self._componentes)

    def __repr__(self):
        return f"Vector({list(self._componentes)})"

    def __str__(self):
        elementos = ", ".join(str(valor) for valor in self._componentes)
        return f"({elementos})"

    def __eq__(self, otro):
        if not isinstance(otro, Vector):
            return NotImplemented
        return self._componentes == otro._componentes

    def __add__(self, otro):
        if not isinstance(otro, Vector):
            return NotImplemented
        if len(self) != len(otro):
            raise ValueError("No se pueden sumar vectores de dimensiones distintas.")
        return Vector(a + b for a, b in zip(self._componentes, otro._componentes))

    def norma(self):
        return sqrt(sum(c ** 2 for c in self._componentes))


u = Vector([1, 2, 3, 4])
v = Vector([4, 3, 2, 1])
print("u =", u, "| repr:", repr(u), "| len(u) =", len(u))
print("u + v =", u + v)
print("¿u + v == Vector([5, 5, 5, 5])?", u + v == Vector([5, 5, 5, 5]))

try:
    u + Vector([1, 2])
except ValueError as error:
    print("Control de dimensión:", error)

In [ ]:
# Ejercicio 3
p = Polinomio([1, 2, 3])
q = Polinomio([4, -2, -3])
suma = p + q

assert suma == Polinomio([5])
assert suma.grado == 0
assert len(suma) == 1

print("P(x) =", p)
print("Q(x) =", q)
print("P(x) + Q(x) =", suma)
print("Grado tras cancelación:", suma.grado, "| len(P + Q):", len(suma))

## 7. Resumen

- `__repr__` ofrece una descripción técnica e inequívoca para depuración, mientras que `__str__` produce la presentación legible usada por `print`.
- `__eq__` define la igualdad matemática de contenido y `__lt__` habilita el ordenamiento natural en `sorted`, `min` y `max`.
- `__add__` permite expresar sumas con el operador `+` y debe devolver `NotImplemented` ante tipos incompatibles.
- `__len__` conecta objetos compuestos como `Vector` o `Polinomio` con la función integrada `len`.
- En el siguiente video organizaremos familias de objetos matemáticos mediante herencia, `super()`, sobrescritura y orden de resolución de métodos.